# 03 · Recovery Models (Regression)

Predicts the recovery rate `rec_rate`. In the LGD formula of the thesis,
`LGD = (1 − cure rate) × (1 − recovery rate)`, the recovery rate only matters for loans that do **not** cure,
so the models are trained and evaluated on the **non-cured loans**. They still predict a recovery rate for
every loan, which notebook 04 combines with P(cure).

| Model | Library / device | Tuning |
|---|---|---|
| Baseline (training mean) | scikit-learn | – |
| Linear Regression | scikit-learn | – |
| Fractional Logit | **PyTorch on MPS** | early stopping on validation |
| Beta Regression | **PyTorch on MPS** | early stopping on validation |
| Zero-One Inflated Beta Regression | **PyTorch on MPS** | early stopping on validation |
| Random Forest | scikit-learn (CPU) | Optuna, 5-fold CV MSE on train |
| XGBoost | xgboost (CPU) | Optuna, 5-fold CV MSE on train; boosting rounds by CV early stopping |
| XGBoost (0/1-inflated) – two-part model | xgboost (CPU) | tuned XGBoost settings |
| Neural Network (tuned) | **PyTorch on MPS** | Optuna on the **validation** set |
| LightGBM → Neural Network stacking ("DeepGBM" in the thesis) | LightGBM (CPU) + **PyTorch on MPS** | out-of-fold LightGBM predictions |
| Ensemble (top 3) | average of the 3 best models by repeated cross-validation | – |

Fractional logit and beta regression are the standard baselines for rates in [0, 1] in the LGD literature.

The validation set holds only a few hundred recovery rates, too few to choose a model reliably. All models are
therefore also compared by **repeated cross-validation** on the training and validation loans; notebook 04
uses that comparison to pick the recovery model.

> **Quick mode.** Start Jupyter with `LGD_QUICK=1` to run every model with 3 trials / 3 epochs. That only
> checks that the notebook runs; the results are meaningless. Leave it unset for real results.

In [ ]:
# Reload the lgd package automatically when its files change (no kernel restart needed after edits).
%load_ext autoreload
%autoreload 2

import time

import numpy as np
import optuna
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MinMaxScaler
from xgboost import XGBClassifier, XGBRegressor

from lgd import torch_models as tm
from lgd.config import (CV_FOLDS, MAX_EPOCHS, N_TRIALS, N_TRIALS_NN, OUTPUT_DIR, PATIENCE, PREDICTION_DIR,
                        QUICK, SEED)
from lgd.data import feature_columns, load_model_dataset, make_splits
from lgd.models import StackedRegressor, ZeroOneInflatedRegressor
from lgd.metrics import bootstrap_ci, format_ci, regression_metrics, results_table
from lgd.plots import (plot_coefficients, plot_history, plot_optuna_history, plot_predicted_vs_actual,
                       plot_residuals, plot_shap_summary, save_and_show)
from lgd.tuning import fit_xgboost_with_early_stopping, tune_network, tune_with_cv, tuning_rows

print(f"Quick mode: {QUICK} | PyTorch device: {tm.DEVICE}")

## Data

`FEATURE_SET = "at_default"` uses only what is known when the loan defaults; `"thesis"` uses every feature.
The models are trained on non-cured loans with a **known** recovery rate.

In [ ]:
FEATURE_SET = "at_default"

dataset = load_model_dataset()
all_loans = make_splits(dataset, "rec_rate", FEATURE_SET)                               # for the predictions
has_recovery = (dataset["cure"] == 0) & dataset["rec_rate"].notna()
data = make_splits(dataset, "rec_rate", FEATURE_SET, rows=has_recovery)                # for training / evaluation
train, valid, test = data["train"], data["valid"], data["test"]
n_features = train.X.shape[1]

scaler = MinMaxScaler().fit(train.X)
X_train_nn, X_valid_nn = scaler.transform(train.X), scaler.transform(valid.X)
tune_rows = tuning_rows(train.y, classification=False)

print(f"{n_features} features | non-cured loans with a recovery rate: train={len(train)}  valid={len(valid)}  test={len(test)}")
train.y.describe()

In [ ]:
predictions = {}    # model -> {"valid": rec_rate, "test": rec_rate} for ALL loans of the split
test_results = {}   # model -> metrics on the non-cured test loans


def record(model_name: str, predict, train_time: float) -> None:
    """Store predictions of a fitted model for all validation / test loans and print its test metrics."""
    predictions[model_name] = {name: pd.Series(np.asarray(predict(all_loans[name].X)).ravel(), index=all_loans[name].X.index)
                               for name in ["valid", "test"]}
    metrics = regression_metrics(test.y, predictions[model_name]["test"].loc[test.y.index])
    test_results[model_name] = {**metrics, "Training Time (s)": train_time}
    print(pd.Series(test_results[model_name]).round(5).to_string())


def predict_network(network):
    return lambda X: tm.predict(network, scaler.transform(X))

# Baseline

In [ ]:
baseline = DummyRegressor(strategy="mean").fit(train.X, train.y)
record("Baseline", baseline.predict, 0.0)

# Linear Regression

In [ ]:
linear_model = make_pipeline(MinMaxScaler(), LinearRegression())
start = time.perf_counter()
linear_model.fit(train.X, train.y)
record("Linear Regression", linear_model.predict, time.perf_counter() - start)

print(f"Intercept: {linear_model[-1].intercept_:.4f}")
plot_coefficients(pd.Series(linear_model[-1].coef_, index=train.X.columns),
                  "Linear Regression - Largest Coefficients", "recovery_linear_coefficients.png")

# Fractional Logit (PyTorch, Apple GPU)

`E[rec_rate | x] = sigmoid(w·x + b)`, fitted by minimising the Bernoulli log-likelihood with the fractional
target (Papke & Wooldridge, 1996). Predictions always lie in [0, 1].

In [ ]:
tm.set_seed(SEED)
fractional_logit = tm.MLP(n_features, hidden=(), loss="bce")
fractional_history = tm.train_network(fractional_logit, X_train_nn, train.y, X_valid_nn, valid.y, learning_rate=0.01,
                                      batch_size=256, max_epochs=MAX_EPOCHS, patience=PATIENCE, verbose=True)
record("Fractional Logit", predict_network(fractional_logit), fractional_history["train_seconds"])

# Beta Regression (PyTorch, Apple GPU)

`rec_rate ~ Beta(μφ, (1 − μ)φ)` with `logit(μ) = w·x + b` and a single precision parameter `φ`,
fitted by maximum likelihood.

In [ ]:
tm.set_seed(SEED)
beta_regression = tm.BetaRegression(n_features, n_observations=len(train))
beta_history = tm.train_network(beta_regression, X_train_nn, train.y, X_valid_nn, valid.y, learning_rate=0.01,
                                batch_size=256, max_epochs=MAX_EPOCHS, patience=PATIENCE, verbose=True)
print(f"Precision phi = {beta_regression.log_phi.exp().item():.2f}")
record("Beta Regression", predict_network(beta_regression), beta_history["train_seconds"])

# Zero-One Inflated Beta Regression (PyTorch, Apple GPU)

A Beta distribution only lives strictly between 0 and 1, but in real LGD data some recoveries are exactly 0 or
exactly 1. Squeezing those values inside (0, 1) makes them dominate the likelihood, so a plain beta regression
fits poorly on such data. The inflated version (Ospina & Ferrari, 2010) models
P(y = 0), P(y = 1) and P(0 < y < 1) separately and uses a Beta distribution only for the partial recoveries.

In [ ]:
tm.set_seed(SEED)
inflated_beta = tm.InflatedBetaRegression(n_features)
inflated_beta_history = tm.train_network(inflated_beta, X_train_nn, train.y, X_valid_nn, valid.y, learning_rate=0.01,
                                         batch_size=256, max_epochs=MAX_EPOCHS, patience=PATIENCE, verbose=True)
print(f"Precision phi = {inflated_beta.log_phi.exp().item():.2f}")
record("Inflated Beta Regression", predict_network(inflated_beta), inflated_beta_history["train_seconds"])

# Random Forest

Optuna, objective: 5-fold CV MSE on the training set.

In [ ]:
def build_random_forest(trial):
    return RandomForestRegressor(
        n_estimators=trial.suggest_int("n_estimators", 50, 400, step=50),
        max_depth=trial.suggest_int("max_depth", 4, 40),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 10),
        max_features=trial.suggest_float("max_features", 0.05, 0.6),
        bootstrap=trial.suggest_categorical("bootstrap", [True, False]),
        n_jobs=-1, random_state=SEED,
    )


rf_study, rf_model, rf_time = tune_with_cv(build_random_forest, train.X, train.y, scoring="neg_mean_squared_error",
                                           n_trials=N_TRIALS, classification=False, study_name="recovery_random_forest")
print("Best parameters:", rf_study.best_params)
plot_optuna_history(rf_study, "Random Forest - Optuna", "−CV MSE", "recovery_random_forest_optuna.png")
record("Random Forest", rf_model.predict, rf_time)

# XGBoost

In [ ]:
def build_xgboost(trial):
    return XGBRegressor(
        n_estimators=trial.suggest_int("n_estimators", 50, 1000, step=50),
        max_depth=trial.suggest_int("max_depth", 1, 10),
        learning_rate=trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        min_child_weight=trial.suggest_int("min_child_weight", 1, 10),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
        objective="reg:squarederror", n_jobs=-1, random_state=SEED,
    )


xgb_study, xgb_model, xgb_time = tune_with_cv(build_xgboost, train.X, train.y, scoring="neg_mean_squared_error",
                                              n_trials=N_TRIALS, classification=False, study_name="recovery_xgboost")
print("Best parameters:", xgb_study.best_params)
plot_optuna_history(xgb_study, "XGBoost - Optuna", "−CV MSE", "recovery_xgboost_optuna.png")
xgb_model, xgb_rounds, xgb_time = fit_xgboost_with_early_stopping(xgb_model, train.X, train.y, classification=False)
print(f"Boosting rounds: {xgb_rounds} (CV early stopping) instead of {xgb_study.best_params['n_estimators']} (Optuna)")
record("XGBoost", xgb_model.predict, xgb_time)

# Two-Part (0/1-Inflated) XGBoost

The same idea as the inflated beta regression with gradient boosting: an XGBoost classifier predicts whether the
recovery is none, full or partial, and an XGBoost regressor predicts the partial recovery rate.
Prediction: P(full) + P(partial) × E[recovery | partial]. Both parts use the tuned XGBoost settings.

In [ ]:
xgb_settings = {**{k: v for k, v in xgb_study.best_params.items() if k != "n_estimators"},
                "n_estimators": xgb_rounds, "n_jobs": -1, "random_state": SEED}
inflated_xgb = ZeroOneInflatedRegressor(XGBClassifier(**xgb_settings), XGBRegressor(**xgb_settings))
start = time.perf_counter()
inflated_xgb.fit(train.X, train.y)
record("XGBoost (0/1-inflated)", inflated_xgb.predict, time.perf_counter() - start)

# Neural Network – Tuned with Optuna (PyTorch, Apple GPU)

Same search space as the thesis (number of layers, neurons, learning rate, batch size, dropout) plus weight
decay. Sigmoid output and MSE loss, as in the thesis. Every trial is scored on the **validation set**.

In [ ]:
def suggest_network(trial):
    network = tm.MLP(n_features,
                     hidden=(trial.suggest_int("num_neurons", 4, 256, log=True),) * trial.suggest_int("num_layers", 1, 8),
                     dropout=trial.suggest_float("dropout_rate", 0.0, 0.5), loss="mse")
    fit_options = {
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-1, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [32, 64, 128, 256, 512]),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
    }
    return network, fit_options


def network_objective(trial):
    tm.set_seed(SEED)
    network, fit_options = suggest_network(trial)
    history = tm.train_network(network, X_train_nn[tune_rows], train.y.iloc[tune_rows], X_valid_nn, valid.y,
                               max_epochs=MAX_EPOCHS, patience=PATIENCE, trial=trial, **fit_options)
    return history["best_val_loss"]


nn_study = tune_network(network_objective, N_TRIALS_NN, "recovery_neural_network")
print("Best parameters:", nn_study.best_params)
plot_optuna_history(nn_study, "Neural Network - Optuna", "Validation MSE", "recovery_nn_optuna.png")

In [ ]:
tm.set_seed(SEED)
tuned_network, fit_options = suggest_network(optuna.trial.FixedTrial(nn_study.best_params))
tuned_history = tm.train_network(tuned_network, X_train_nn, train.y, X_valid_nn, valid.y, max_epochs=MAX_EPOCHS,
                                 patience=PATIENCE, verbose=True, **fit_options)
plot_history(tuned_history, "Neural Network (tuned)", "recovery_nn_tuned_training_curves.png")
record("Neural Network (tuned)", predict_network(tuned_network), tuned_history["train_seconds"])

# LightGBM → Neural Network Stacking ("DeepGBM" in the thesis)

1. LightGBM (thesis parameters) predicts the recovery rate.
2. That prediction is appended to the scaled features and a 64 → 32 → 1 PyTorch network is trained on them.

For the training rows the LightGBM column comes from **5-fold out-of-fold predictions**, i.e. from models that
did not see those rows; otherwise it would look far more accurate on the training set than on new loans,
and the network would learn to trust it too much. Validation and test rows get the prediction of a LightGBM
model fitted on the whole training set.

*Note:* DeepGBM (Ke et al., 2019) is a different, more involved architecture; "stacking" is the accurate name here.

In [ ]:
def build_lightgbm():
    return LGBMRegressor(objective="regression", learning_rate=0.05, num_leaves=31, n_estimators=100,
                         force_row_wise=True, random_state=SEED, verbose=-1)


start = time.perf_counter()
lgbm_out_of_fold = cross_val_predict(build_lightgbm(), train.X, train.y,
                                     cv=KFold(CV_FOLDS, shuffle=True, random_state=SEED))
lgbm_model = build_lightgbm().fit(train.X, train.y)
lgbm_time = time.perf_counter() - start

importance = pd.Series(lgbm_model.booster_.feature_importance("gain"), index=train.X.columns).nlargest(10)[::-1]
ax = importance.plot.barh(figsize=(8, 5), title="LightGBM Feature Importances (gain)")
save_and_show(ax.figure, "recovery_lightgbm_feature_importance.png")

In [ ]:
def stacked_features(X, lgbm_prediction=None):
    lgbm_prediction = lgbm_model.predict(X) if lgbm_prediction is None else lgbm_prediction
    return np.column_stack([scaler.transform(X), lgbm_prediction])


tm.set_seed(SEED)
stacking_network = tm.MLP(n_features + 1, hidden=(64, 32), loss="mse")
stacking_history = tm.train_network(stacking_network, stacked_features(train.X, lgbm_out_of_fold), train.y,
                                    stacked_features(valid.X), valid.y, learning_rate=0.001, batch_size=32,
                                    max_epochs=MAX_EPOCHS, patience=PATIENCE, verbose=True)
plot_history(stacking_history, "LightGBM → Neural Network stacking", "recovery_stacking_training_curves.png")

record("LightGBM → NN Stacking", lambda X: tm.predict(stacking_network, stacked_features(X)),
       lgbm_time + stacking_history["train_seconds"])

# Model Selection by Repeated Cross-Validation

Every model is refitted with its tuned settings in 5-fold cross-validation, repeated 3 times with different
folds, on the training **and** validation loans together (about 4,000 recovery rates instead of ~500 validation
rates). The PyTorch models keep 15% of each training fold for early stopping. The ensemble averages the
cross-validated predictions of the three best models. Notebook 04 picks the recovery model with the lowest
cross-validated MSE.

In [ ]:
X_development, y_development = pd.concat([train.X, valid.X]), pd.concat([train.y, valid.y])
torch_settings = {"max_epochs": MAX_EPOCHS, "patience": PATIENCE}
candidates = {
    "Baseline": DummyRegressor(strategy="mean"),
    "Linear Regression": make_pipeline(MinMaxScaler(), LinearRegression()),
    "Fractional Logit": tm.TorchRegressor(lambda n, m: tm.MLP(n, hidden=(), loss="bce"),
                                          {"learning_rate": 0.01, "batch_size": 256}, **torch_settings),
    "Beta Regression": tm.TorchRegressor(lambda n, m: tm.BetaRegression(n, m),
                                         {"learning_rate": 0.01, "batch_size": 256}, **torch_settings),
    "Inflated Beta Regression": tm.TorchRegressor(lambda n, m: tm.InflatedBetaRegression(n),
                                                  {"learning_rate": 0.01, "batch_size": 256}, **torch_settings),
    "Random Forest": clone(rf_model),
    "XGBoost": clone(xgb_model),
    "XGBoost (0/1-inflated)": clone(inflated_xgb),
    "Neural Network (tuned)": tm.TorchRegressor(lambda n, m: suggest_network(optuna.trial.FixedTrial(nn_study.best_params))[0],
                                                fit_options, **torch_settings),
    "LightGBM → NN Stacking": StackedRegressor(build_lightgbm(), tm.TorchRegressor(
        lambda n, m: tm.MLP(n, hidden=(64, 32), loss="mse"), {"learning_rate": 0.001, "batch_size": 32}, **torch_settings)),
}

N_REPEATS = 1 if QUICK else 3
out_of_fold = {name: [] for name in candidates}
start = time.perf_counter()
for repeat in range(N_REPEATS):
    folds = KFold(CV_FOLDS, shuffle=True, random_state=SEED + repeat)
    for name, estimator in candidates.items():
        out_of_fold[name].append(cross_val_predict(estimator, X_development, y_development, cv=folds))
print(f"Repeated cross-validation took {time.perf_counter() - start:.0f}s")

ranking = sorted((name for name in candidates if name != "Baseline"),
                 key=lambda name: np.mean([mean_squared_error(y_development, p) for p in out_of_fold[name]]))
ensemble_members = ranking[:3]
out_of_fold["Ensemble (top 3)"] = [np.mean([out_of_fold[name][r] for name in ensemble_members], axis=0) for r in range(N_REPEATS)]
print("Ensemble members:", ensemble_members)

model_selection = pd.DataFrame({
    name: {"CV MSE": np.mean([mean_squared_error(y_development, p) for p in preds]),
           "CV MSE std": np.std([mean_squared_error(y_development, p) for p in preds]),
           "CV R^2": np.mean([r2_score(y_development, p) for p in preds])}
    for name, preds in out_of_fold.items()}).T.sort_values("CV MSE")
model_selection.to_csv(OUTPUT_DIR / "recovery_model_selection_cv.csv")

predictions["Ensemble (top 3)"] = {split: sum(predictions[name][split] for name in ensemble_members) / len(ensemble_members)
                                   for split in ["valid", "test"]}
test_results["Ensemble (top 3)"] = {**regression_metrics(test.y, predictions["Ensemble (top 3)"]["test"].loc[test.y.index]),
                                    "Training Time (s)": sum(test_results[name]["Training Time (s)"] for name in ensemble_members)}
print(f"Best recovery model by repeated CV: {model_selection.index[0]}")
model_selection.round(5)

# Comparison (non-cured test loans)

95% bootstrap confidence intervals (resampling the test loans) show whether two models really differ.

In [ ]:
test_predictions = {name: p["test"].loc[test.y.index] for name, p in predictions.items()}

summary = results_table(test_results)
summary["MSE 95% CI"] = [format_ci(summary.loc[name, "MSE"], bootstrap_ci(test.y, p, mean_squared_error), digits=5)
                         for name, p in test_predictions.items()]
summary["R^2 95% CI"] = [format_ci(summary.loc[name, "R^2"], bootstrap_ci(test.y, p, r2_score))
                         for name, p in test_predictions.items()]
summary

In [ ]:
models_to_plot = {name: p for name, p in test_predictions.items() if name != "Baseline"}
plot_residuals(test.y, models_to_plot, "recovery_residuals.png")
plot_predicted_vs_actual(test.y, models_to_plot, "recovery_predicted_vs_actual.png")

# Interpretation – SHAP Values (XGBoost)

In [ ]:
plot_shap_summary(xgb_model, test.X, "XGBoost - SHAP values for the recovery rate", "recovery_xgboost_shap.png")

# Effect of the Post-Default Features

Linear Regression and XGBoost (with the tuned parameters) trained on both feature sets.

In [ ]:
if feature_columns(dataset, "thesis") == feature_columns(dataset, "at_default"):
    print("This dataset has no post-default features - nothing to compare.")
else:
    feature_set_data = {name: make_splits(dataset, "rec_rate", name, rows=has_recovery) for name in ["at_default", "thesis"]}
    model_builders = {
        "Linear Regression": lambda: make_pipeline(MinMaxScaler(), LinearRegression()),
        "XGBoost": lambda: XGBRegressor(**xgb_study.best_params, objective="reg:squarederror", n_jobs=-1, random_state=SEED),
    }

    feature_set_comparison = {}
    for model_name, build in model_builders.items():
        for set_name, splits in feature_set_data.items():
            model = build().fit(splits["train"].X, splits["train"].y)
            feature_set_comparison[(model_name, set_name)] = regression_metrics(splits["test"].y, model.predict(splits["test"].X))

    display(pd.DataFrame(feature_set_comparison).T)

# Save Results and Predictions (used by notebook 04)

In [ ]:
validation_results = results_table({name: regression_metrics(valid.y, p["valid"].loc[valid.y.index])
                                    for name, p in predictions.items()})
summary.to_csv(OUTPUT_DIR / "recovery_results_test.csv")
validation_results.to_csv(OUTPUT_DIR / "recovery_results_valid.csv")

for split_name in ["valid", "test"]:
    pd.DataFrame({name: p[split_name] for name, p in predictions.items()}).to_csv(PREDICTION_DIR / f"recovery_{split_name}.csv")

validation_results